# Bound-Retention Model Training

This notebook trains and evaluates ML models for **rapidly screening new asteroid impact scenarios before expensive SPH simulations**.

The primary objective is now to predict **continuous physical outcomes**, especially `bound_mass_fraction`, rather than first solving a binary classification problem. The workflow uses grouped cross-validation to:

1. train a default regression model for `bound_mass_fraction` and analyse threshold behaviour around `BMF = 10%`
2. train supporting regressors for secondary bound-retention targets
3. train scientifically useful fragmentation regressors that help interpret impact outcomes
4. derive fragmentation regimes and SPH decision-support rules from the data and model errors

## Structure

1. **Default target: `bound_mass_fraction` regression**
2. **Secondary/supporting bound regressions**
3. **Fragmentation regressions**
4. **Fragmentation Regime Analysis**
5. **SPH Decision Support**
6. **Parameter-Space Condition Analysis**
7. **Future Work: Interactive Decision Dashboard**


## 0 · Imports and configuration

In [ ]:
from __future__ import annotations

import pickle
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

RANDOM_STATE = 42
N_SPLITS = 5
DEFAULT_RF_R2_TOL = 0.02
BMF_DECISION_THRESHOLD = 0.10

DATASET_PATH = Path("outputs/bound_outcomes.csv")
SAVE_DIR = Path("ml/bound_outcomes/notebook_demo")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_TARGET = "bound_mass_fraction"
BOUND_SUPPORT_TARGETS = [
    "bound_fragment_count",
    "largest_bound_fragment_mass_kg",
    "average_bound_fragment_mass_kg",
]

FRAGMENTATION_BASE_TARGETS = [
    "n_fragments",
    "largest_fragment_mass_kg",
]
FRAGMENTATION_PARTICLE_CANDIDATES = [
    "largest_fragment_particles",
    "largest_fragment_particle_count",
    "largest_fragment_n_particles",
    "largest_fragment_particle_number",
]

FEATURE_COLUMNS = [
    "mass_log10_kg",
    "particle_log10",
    "periapsis_Rm",
    "v_inf_kms",
    "spin_period_hr",
    "spin_axis",
    "has_explicit_spin",
    "special_case_code",
    "timestep",
    "fof_linking_length",
]

plt.style.use("seaborn-v0_8-whitegrid")
print("Configuration ready.")


## 1 · Filename parser and feature engineering

In [ ]:
FILENAME_RE = re.compile(
    r"^(?P<prefix>Ma_xp)_(?P<mass>A\d{4}(?:c30)?)(?:_(?P<spin>s\d{3}[A-Za-z]*))?_n(?P<resolution>\d+)"
    r"_r(?P<periapsis>\d+)_v(?P<velocity>\d+)_(?P<timestep>\d+)_fof_(?P<linking_length>[0-9.]+)_(?P<chunk>\d+)\.hdf5$"
)


def parse_simulation_filename(filename: str) -> dict[str, object]:
    match = FILENAME_RE.match(filename)
    if not match:
        raise ValueError(f"Unrecognized FoF filename pattern: {filename}")

    mass_code = match.group("mass")
    spin_code = match.group("spin") or ""
    special_case_code = "c30" if mass_code.endswith("c30") else ""
    mass_digits = mass_code[1:5]
    spin_axis = spin_code[4:] if len(spin_code) > 4 else ""
    spin_value = spin_code[1:4] if spin_code else ""

    return {
        "filename": filename,
        "mass_code": mass_code,
        "mass_value": int(mass_digits),
        "special_case_code": special_case_code,
        "spin_code": spin_code,
        "spin_value": int(spin_value) if spin_value else "",
        "spin_axis": spin_axis,
        "has_explicit_spin": bool(spin_code),
        "resolution_code": f"n{int(match.group('resolution'))}",
        "resolution_value": int(match.group("resolution")),
        "periapsis_value": int(match.group("periapsis")),
        "velocity_value": int(match.group("velocity")),
        "timestep": int(match.group("timestep")),
        "fof_linking_length": float(match.group("linking_length")),
        "chunk_index": int(match.group("chunk")),
    }


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy()
    parsed = frame["fof_file"].map(parse_simulation_filename).apply(pd.Series)
    for column in parsed.columns:
        if column not in frame.columns:
            frame[column] = parsed[column]

    frame["mass_log10_kg"] = pd.to_numeric(frame["mass_value"], errors="coerce") / 100.0
    frame["target_mass_kg"] = 10 ** frame["mass_log10_kg"]

    resolution_values = pd.to_numeric(frame["resolution_value"], errors="coerce")
    frame["particle_log10"] = resolution_values.map(lambda x: np.nan if pd.isna(x) else np.log10(x))
    frame["periapsis_Rm"] = pd.to_numeric(frame["periapsis_value"], errors="coerce") / 10.0
    frame["v_inf_kms"] = pd.to_numeric(frame["velocity_value"], errors="coerce") / 10.0
    frame["spin_period_hr"] = pd.to_numeric(frame["spin_value"], errors="coerce") / 10.0
    frame["has_explicit_spin"] = (
        frame["has_explicit_spin"].fillna(False).astype(str).str.lower().isin({"true", "1", "yes"})
    )
    frame["spin_axis"] = frame["spin_axis"].fillna("none").replace("", "none")
    frame["special_case_code"] = frame["special_case_code"].fillna("").replace("", "none")

    bound_mass_kg = pd.to_numeric(frame["bound_mass_kg"], errors="coerce")
    bound_fragment_count = pd.to_numeric(frame["bound_fragment_count"], errors="coerce").replace(0, np.nan)
    frame["average_bound_fragment_mass_kg"] = bound_mass_kg / bound_fragment_count

    largest_bound = pd.to_numeric(frame["largest_bound_fragment_mass_kg"], errors="coerce").fillna(0)
    largest_unbound = pd.to_numeric(frame["largest_unbound_fragment_mass_kg"], errors="coerce").fillna(0)
    frame["largest_fragment_mass_kg"] = largest_bound.where(largest_bound >= largest_unbound, largest_unbound)
    frame["largest_fragment_mass_fraction"] = frame["largest_fragment_mass_kg"] / frame["target_mass_kg"]

    return frame


print("Feature engineering functions defined.")


## 2 · Load and prepare data

In [ ]:
raw_df = pd.read_csv(DATASET_PATH, low_memory=False)
df = add_engineered_features(raw_df)

X = df[FEATURE_COLUMNS].copy()
groups = df["physical_file"].astype(str)

available_particle_targets = [c for c in FRAGMENTATION_PARTICLE_CANDIDATES if c in df.columns]
FRAGMENTATION_TARGETS = FRAGMENTATION_BASE_TARGETS + available_particle_targets[:1]

FRAGMENTATION_TARGET_DESCRIPTIONS = {
    "n_fragments": "Total fragment count produced by the collision.",
    "largest_fragment_mass_kg": "Mass of the single largest fragment, regardless of whether it remains bound.",
}
if available_particle_targets:
    FRAGMENTATION_TARGET_DESCRIPTIONS[available_particle_targets[0]] = (
        "Largest-fragment particle count available directly in the dataset."
    )

excluded_fragment_targets = []
if "total_fragment_mass_kg" in df.columns:
    total_mass = pd.to_numeric(df["total_fragment_mass_kg"], errors="coerce")
    corr_with_target_mass = total_mass.corr(df["target_mass_kg"])
    excluded_fragment_targets.append({
        "target": "total_fragment_mass_kg",
        "status": "Excluded from main modelling",
        "reason": (
            "More useful as a mass-budget diagnostic than as a screening target; "
            f"correlation with inferred target mass is {corr_with_target_mass:.3f}."
        ),
    })

summary_rows = []
for target in [PRIMARY_TARGET] + BOUND_SUPPORT_TARGETS + FRAGMENTATION_TARGETS:
    col = pd.to_numeric(df[target], errors="coerce")
    summary_rows.append({
        "target": target,
        "non_null": int(col.notna().sum()),
        "mean": float(col.mean()),
        "std": float(col.std()),
        "min": float(col.min()),
        "max": float(col.max()),
    })

print(f"Rows loaded: {len(df)}")
print(f"Unique grouped simulations: {groups.nunique()}")
if available_particle_targets:
    print(f"Including largest-fragment particle descriptor: {available_particle_targets[0]}")
else:
    print("No largest-fragment particle-count target found in the dataset; using mass-based fragmentation descriptors only.")

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(6))

if excluded_fragment_targets:
    display(pd.DataFrame(excluded_fragment_targets))


## 3 · Preprocessing and model builders

In [ ]:
CATEGORICAL_FEATURES = [c for c in ["spin_axis", "special_case_code"] if c in FEATURE_COLUMNS]
NUMERIC_FEATURES = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_FEATURES]


def build_preprocessor(scale: bool = False) -> ColumnTransformer:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(numeric_steps), NUMERIC_FEATURES),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                CATEGORICAL_FEATURES,
            ),
        ]
    )


def build_regressors() -> dict[str, Pipeline]:
    return {
        "ridge": Pipeline([
            ("preprocessor", build_preprocessor(scale=True)),
            ("model", Ridge()),
        ]),
        "random_forest": Pipeline([
            ("preprocessor", build_preprocessor()),
            ("model", RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "gradient_boosting": Pipeline([
            ("preprocessor", build_preprocessor()),
            ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
        ]),
    }


print("Model builders ready.")


## 4 · Evaluation helpers

In [ ]:
def reg_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = np.array(y_true, dtype=float)[mask]
    y_pred = np.array(y_pred, dtype=float)[mask]
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "n": int(len(y_true)),
    }


def evaluate_regressor(name, base_pipeline, X, y, groups):
    splitter = GroupKFold(n_splits=min(N_SPLITS, groups.nunique()))
    y_pred = pd.Series(np.nan, index=y.index, dtype="float64")

    valid_mask = y.notna()
    X_v, y_v, g_v = X.loc[valid_mask], y.loc[valid_mask], groups.loc[valid_mask]

    for train_idx, test_idx in splitter.split(X_v, y_v, g_v):
        pipeline = clone(base_pipeline)
        pipeline.fit(X_v.iloc[train_idx], y_v.iloc[train_idx])
        fold_pred = pipeline.predict(X_v.iloc[test_idx])
        y_pred.iloc[X_v.index[test_idx]] = fold_pred

    oof_metrics = reg_metrics(y_v.to_numpy(), y_pred.loc[valid_mask].to_numpy())

    final = clone(base_pipeline)
    final.fit(X_v, y_v)
    train_pred = final.predict(X_v)
    train_metrics = reg_metrics(y_v.to_numpy(), train_pred)

    row = {"model": name, **oof_metrics, **{f"train_{k}": v for k, v in train_metrics.items()}}
    preds = pd.DataFrame({"actual": y_v, "predicted": y_pred.loc[valid_mask]}, index=y_v.index)
    preds["residual"] = preds["predicted"] - preds["actual"]
    preds["abs_error"] = preds["residual"].abs()
    return row, preds, final


def select_default_model(metrics_df: pd.DataFrame, target: str, tol: float = DEFAULT_RF_R2_TOL):
    target_metrics = metrics_df.query("target == @target").sort_values(["r2", "mae"], ascending=[False, True]).copy()
    best_model = target_metrics.iloc[0]["model"]
    best_r2 = float(target_metrics.iloc[0]["r2"])

    rf_row = target_metrics[target_metrics["model"] == "random_forest"]
    if not rf_row.empty and float(rf_row.iloc[0]["r2"]) >= best_r2 - tol:
        return "random_forest", f"Random Forest is within {tol:.3f} OOF R² of the best model, so it is kept as the default baseline."

    return best_model, f"{best_model} has the strongest OOF R² for this target and becomes the default model."


def plot_actual_vs_predicted_grid(pred_map, metrics_df, target_order, title, color):
    model_order = ["ridge", "random_forest", "gradient_boosting"]
    fig, axes = plt.subplots(len(target_order), len(model_order), figsize=(5 * len(model_order), 4 * len(target_order)))
    if len(target_order) == 1:
        axes = np.array([axes])

    for t_idx, target in enumerate(target_order):
        for m_idx, model_name in enumerate(model_order):
            ax = axes[t_idx, m_idx]
            pf = pred_map[target][model_name].dropna()
            ax.scatter(pf["actual"], pf["predicted"], alpha=0.45, s=18, color=color)
            lim_min = float(min(pf["actual"].min(), pf["predicted"].min()))
            lim_max = float(max(pf["actual"].max(), pf["predicted"].max()))
            ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", lw=1)
            r = metrics_df.query("target == @target and model == @model_name").iloc[0]
            ax.set_title(f"{target}\n{model_name} | R²={r['r2']:.3f}, MAE={r['mae']:.3e}, RMSE={r['rmse']:.3e}")
            ax.set_xlabel("Actual")
            ax.set_ylabel("Predicted (OOF)")

    fig.suptitle(title, fontsize=14, y=1.01)
    fig.tight_layout()
    plt.show()


print("Evaluation helpers ready.")


PARAMETER_NUMERIC_COLUMNS = [
    "mass_log10_kg",
    "periapsis_Rm",
    "v_inf_kms",
    "spin_period_hr",
    "particle_log10",
    "fof_linking_length",
]

PARAMETER_CATEGORICAL_COLUMNS = [
    "spin_axis",
    "has_explicit_spin",
    "special_case_code",
]


def summarise_parameter_space(frame, group_col):
    numeric_summary = (
        frame.groupby(group_col)[PARAMETER_NUMERIC_COLUMNS]
        .agg(["min", lambda s: s.quantile(0.25), "median", lambda s: s.quantile(0.75), "max"])
        .rename(columns={"<lambda_0>": "q25", "<lambda_1>": "q75"})
    )
    categorical_frames = []
    for cat_col in PARAMETER_CATEGORICAL_COLUMNS:
        counts = (
            frame.groupby(group_col)[cat_col]
            .value_counts(dropna=False, normalize=True)
            .rename("share")
            .reset_index()
        )
        counts["count"] = (
            frame.groupby(group_col)[cat_col]
            .value_counts(dropna=False)
            .reset_index(name="count")["count"]
        )
        counts.insert(1, "parameter", cat_col)
        counts = counts.rename(columns={cat_col: "category"})
        categorical_frames.append(counts.sort_values([group_col, "parameter", "share"], ascending=[True, True, False]))
    categorical_summary = pd.concat(categorical_frames, ignore_index=True)
    return numeric_summary, categorical_summary


def plot_parameter_pairs(frame, group_col, title_prefix):
    pair_specs = [
        ("periapsis_Rm", "v_inf_kms"),
        ("mass_log10_kg", "periapsis_Rm"),
        ("spin_period_hr", "periapsis_Rm"),
        ("mass_log10_kg", "v_inf_kms"),
    ]
    labels = list(pd.Series(frame[group_col]).dropna().unique())
    cmap = plt.get_cmap("tab10")
    color_map = {label: cmap(i % 10) for i, label in enumerate(labels)}

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for ax, (x_col, y_col) in zip(axes.flat, pair_specs):
        for label in labels:
            subset = frame[frame[group_col] == label]
            ax.scatter(subset[x_col], subset[y_col], s=26, alpha=0.65, label=label, color=color_map[label])
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title(f"{title_prefix}: {y_col} vs {x_col}")
    handles, legend_labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper center", ncol=min(3, len(legend_labels)), frameon=True)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()


def compute_permutation_importance_table(model, X_eval, y_eval, scoring="r2", n_repeats=10):
    result = permutation_importance(
        model,
        X_eval,
        y_eval,
        scoring=scoring,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    importance_df = pd.DataFrame({
        "feature": X_eval.columns,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False)
    return importance_df


def make_rule_summary(frame, group_col, category_name):
    rows = []
    for group_value, subset in frame.groupby(group_col):
        numeric = subset[PARAMETER_NUMERIC_COLUMNS].agg([lambda s: s.quantile(0.25), "median", lambda s: s.quantile(0.75)])
        numeric.index = ["q25", "median", "q75"]
        pieces = []
        for col in ["periapsis_Rm", "v_inf_kms", "mass_log10_kg", "spin_period_hr", "particle_log10", "fof_linking_length"]:
            q25 = numeric.loc["q25", col]
            med = numeric.loc["median", col]
            q75 = numeric.loc["q75", col]
            if pd.notna(q25) and pd.notna(med) and pd.notna(q75):
                pieces.append(f"{col}: median {med:.3f} (IQR {q25:.3f}–{q75:.3f})")
        rows.append({
            category_name: group_value,
            "n_rows": len(subset),
            "rule_summary": "; ".join(pieces),
        })
    return pd.DataFrame(rows)


---
## 5 · Default target: `bound_mass_fraction` regression

This is now the primary modelling task. The notebook predicts the continuous `bound_mass_fraction` first, then performs threshold analysis around `BMF = 10%` using the regression outputs rather than a separate classifier.


In [ ]:
y_primary = pd.to_numeric(df[PRIMARY_TARGET], errors="coerce")

primary_rows = []
primary_preds = {}
primary_trained = {}

print(f"Target: {PRIMARY_TARGET}")
print(f"  Non-null rows: {y_primary.notna().sum()}")
print(f"  Mean BMF     : {y_primary.mean():.4f}")
print(f"  Std BMF      : {y_primary.std():.4f}")

for model_name, pipeline in build_regressors().items():
    print(f"  Training {model_name} …")
    row, preds, fitted = evaluate_regressor(model_name, pipeline, X, y_primary, groups)
    row["target"] = PRIMARY_TARGET
    primary_rows.append(row)
    primary_preds[model_name] = preds
    primary_trained[model_name] = fitted

primary_metrics_df = pd.DataFrame(primary_rows).sort_values(["r2", "mae"], ascending=[False, True])
default_primary_model, primary_default_reason = select_default_model(primary_metrics_df, PRIMARY_TARGET)

print(f"\nBest OOF model     : {primary_metrics_df.iloc[0]['model']}")
print(f"Default baseline   : {default_primary_model}")
print(primary_default_reason)

display(primary_metrics_df[["model", "r2", "mae", "rmse", "n", "train_r2", "train_mae", "train_rmse"]].round(6))


In [ ]:
plot_actual_vs_predicted_grid(
    pred_map={PRIMARY_TARGET: primary_preds},
    metrics_df=primary_metrics_df,
    target_order=[PRIMARY_TARGET],
    title="Primary target — bound_mass_fraction actual vs predicted (OOF)",
    color="darkorange",
)


In [ ]:
primary_default_preds = primary_preds[default_primary_model].copy()
primary_default_preds["predicted_clipped"] = primary_default_preds["predicted"].clip(0, 1)
primary_default_preds["decision_distance"] = primary_default_preds["predicted_clipped"] - BMF_DECISION_THRESHOLD
primary_default_preds["actual_ge_0_1"] = primary_default_preds["actual"] >= BMF_DECISION_THRESHOLD
primary_default_preds["pred_ge_0_1"] = primary_default_preds["predicted_clipped"] >= BMF_DECISION_THRESHOLD

bmf_error_margin = float(primary_default_preds["abs_error"].quantile(0.75))
threshold_accuracy = float((primary_default_preds["actual_ge_0_1"] == primary_default_preds["pred_ge_0_1"]).mean())

primary_default_preds["threshold_region"] = np.select(
    [
        primary_default_preds["predicted_clipped"] < BMF_DECISION_THRESHOLD - bmf_error_margin,
        primary_default_preds["predicted_clipped"] > BMF_DECISION_THRESHOLD + bmf_error_margin,
    ],
    ["Predicted BMF < 10%", "Predicted BMF ≥ 10%"],
    default="Borderline around 10%",
)

threshold_summary = pd.DataFrame([
    {
        "decision_threshold": BMF_DECISION_THRESHOLD,
        "derived_borderline_margin": bmf_error_margin,
        "threshold_agreement_from_regression": threshold_accuracy,
        "borderline_share": float((primary_default_preds["threshold_region"] == "Borderline around 10%").mean()),
    }
])

print(f"Derived borderline margin around BMF = 10%: ±{bmf_error_margin:.4f}")
print(f"Regression threshold agreement at BMF = 10%: {threshold_accuracy:.3f}")
display(threshold_summary.round(6))
display(
    primary_default_preds.groupby("threshold_region")[["actual", "predicted_clipped", "abs_error"]]
    .agg(["count", "mean", "median"]) 
    .round(6)
)

fig, ax = plt.subplots(figsize=(7, 6))
colors = {
    "Predicted BMF < 10%": "firebrick",
    "Borderline around 10%": "goldenrod",
    "Predicted BMF ≥ 10%": "seagreen",
}
for label, group in primary_default_preds.groupby("threshold_region"):
    ax.scatter(group["actual"], group["predicted_clipped"], label=label, alpha=0.65, s=24, color=colors[label])

ax.axvline(BMF_DECISION_THRESHOLD, color="black", ls="--", lw=1)
ax.axhline(BMF_DECISION_THRESHOLD, color="black", ls="--", lw=1)
ax.axhspan(BMF_DECISION_THRESHOLD - bmf_error_margin, BMF_DECISION_THRESHOLD + bmf_error_margin, color="grey", alpha=0.15)
ax.axvspan(BMF_DECISION_THRESHOLD - bmf_error_margin, BMF_DECISION_THRESHOLD + bmf_error_margin, color="grey", alpha=0.08)
ax.set_xlabel("Actual bound_mass_fraction")
ax.set_ylabel(f"Predicted bound_mass_fraction ({default_primary_model}, OOF)")
ax.set_title("Threshold analysis from regression predictions")
ax.legend()
plt.show()


---
## 6 · Secondary/supporting bound regression tasks

These are kept as supporting regressions because they help interpret the bound outcome once `bound_mass_fraction` has been estimated.


In [ ]:
bound_support_rows = []
bound_support_preds = {}
bound_support_trained = {}

for target in BOUND_SUPPORT_TARGETS:
    y_reg = pd.to_numeric(df[target], errors="coerce")
    print(f"\n── Target: {target}  (n={y_reg.notna().sum()}, mean={y_reg.mean():.3e}) ──")

    target_rows, target_preds, target_trained = [], {}, {}
    for model_name, pipeline in build_regressors().items():
        print(f"   {model_name} …")
        row, preds, fitted = evaluate_regressor(model_name, pipeline, X, y_reg, groups)
        row["target"] = target
        target_rows.append(row)
        target_preds[model_name] = preds
        target_trained[model_name] = fitted

    bound_support_rows.extend(target_rows)
    bound_support_preds[target] = target_preds
    bound_support_trained[target] = target_trained

bound_support_metrics_df = pd.DataFrame(bound_support_rows)[
    ["target", "model", "r2", "mae", "rmse", "n", "train_r2", "train_mae", "train_rmse"]
].sort_values(["target", "r2", "mae"], ascending=[True, False, True])

display(bound_support_metrics_df.round(6))


In [ ]:
plot_actual_vs_predicted_grid(
    pred_map=bound_support_preds,
    metrics_df=bound_support_metrics_df,
    target_order=BOUND_SUPPORT_TARGETS,
    title="Supporting bound targets — actual vs predicted (OOF)",
    color="darkorange",
)


---
## 7 · Fragmentation regression tasks

The fragmentation section now prioritises descriptors that are scientifically useful for screening impact outcomes:

- `n_fragments`
- `largest_fragment_mass_kg`
- largest-fragment particle count **if the dataset provides it directly**

`total_fragment_mass_kg` is not included as a main modelling target here because it behaves more like a mass-budget diagnostic than a regime-defining screening output.


In [ ]:
frag_rows = []
frag_preds = {}
frag_trained = {}

for target in FRAGMENTATION_TARGETS:
    y_reg = pd.to_numeric(df[target], errors="coerce")
    print(f"\n── Target: {target}  (n={y_reg.notna().sum()}, mean={y_reg.mean():.3e}) ──")

    target_rows, target_preds, target_trained = [], {}, {}
    for model_name, pipeline in build_regressors().items():
        print(f"   {model_name} …")
        row, preds, fitted = evaluate_regressor(model_name, pipeline, X, y_reg, groups)
        row["target"] = target
        target_rows.append(row)
        target_preds[model_name] = preds
        target_trained[model_name] = fitted

    frag_rows.extend(target_rows)
    frag_preds[target] = target_preds
    frag_trained[target] = target_trained

frag_metrics_df = pd.DataFrame(frag_rows)[
    ["target", "model", "r2", "mae", "rmse", "n", "train_r2", "train_mae", "train_rmse"]
].sort_values(["target", "r2", "mae"], ascending=[True, False, True])

display(frag_metrics_df.round(6))


In [ ]:
plot_actual_vs_predicted_grid(
    pred_map=frag_preds,
    metrics_df=frag_metrics_df,
    target_order=FRAGMENTATION_TARGETS,
    title="Fragmentation targets — actual vs predicted (OOF)",
    color="steelblue",
)


In [ ]:
all_trained_models = {PRIMARY_TARGET: primary_trained, **bound_support_trained, **frag_trained}
saved_models = []
for target, model_dict in all_trained_models.items():
    for model_name, fitted in model_dict.items():
        out = SAVE_DIR / f"{target}__with_fof_linking_length__{model_name}.pkl"
        with out.open("wb") as fh:
            pickle.dump(fitted, fh)
        saved_models.append({"target": target, "model": model_name, "path": str(out)})
        print(f"Saved: {out}")

display(pd.DataFrame(saved_models))


---
## 8 · Fragmentation Regime Analysis

The goal here is to turn the continuous regression outputs into interpretable outcome regimes:

- **Mostly intact**
- **Moderate fragmentation**
- **Strong fragmentation**

The boundaries are derived from the **dataset distributions** and the **OOF prediction errors** rather than being hard-coded by hand.


In [ ]:
default_model_rows = []
default_models = {PRIMARY_TARGET: default_primary_model}

for target in BOUND_SUPPORT_TARGETS:
    model_name, reason = select_default_model(bound_support_metrics_df, target)
    default_models[target] = model_name
    default_model_rows.append({"target": target, "default_model": model_name, "reason": reason})

for target in FRAGMENTATION_TARGETS:
    model_name, reason = select_default_model(frag_metrics_df, target)
    default_models[target] = model_name
    default_model_rows.append({"target": target, "default_model": model_name, "reason": reason})

print(primary_default_reason)
display(pd.DataFrame([{"target": PRIMARY_TARGET, "default_model": default_primary_model, "reason": primary_default_reason}, *default_model_rows]))

analysis_df = df[[
    "physical_file",
    "mass_log10_kg",
    "particle_log10",
    "periapsis_Rm",
    "v_inf_kms",
    "spin_period_hr",
    "spin_axis",
    "has_explicit_spin",
    "special_case_code",
    "timestep",
    "fof_linking_length",
    "target_mass_kg",
    "bound_mass_fraction",
    "n_fragments",
    "largest_fragment_mass_kg",
    "bound_fragment_count",
    "largest_bound_fragment_mass_kg",
    "average_bound_fragment_mass_kg",
]].copy()

analysis_df["pred_bound_mass_fraction"] = primary_preds[default_primary_model]["predicted"].clip(0, 1)
analysis_df["bmf_abs_error"] = primary_preds[default_primary_model]["abs_error"]
analysis_df["pred_bound_fragment_count"] = bound_support_preds["bound_fragment_count"][default_models["bound_fragment_count"]]["predicted"]
analysis_df["pred_largest_bound_fragment_mass_kg"] = bound_support_preds["largest_bound_fragment_mass_kg"][default_models["largest_bound_fragment_mass_kg"]]["predicted"]
analysis_df["pred_average_bound_fragment_mass_kg"] = bound_support_preds["average_bound_fragment_mass_kg"][default_models["average_bound_fragment_mass_kg"]]["predicted"]
analysis_df["pred_n_fragments"] = frag_preds["n_fragments"][default_models["n_fragments"]]["predicted"]
analysis_df["pred_largest_fragment_mass_kg"] = frag_preds["largest_fragment_mass_kg"][default_models["largest_fragment_mass_kg"]]["predicted"]
analysis_df["pred_largest_fragment_mass_fraction"] = analysis_df["pred_largest_fragment_mass_kg"] / analysis_df["target_mass_kg"]
analysis_df["actual_largest_fragment_mass_fraction"] = analysis_df["largest_fragment_mass_kg"] / analysis_df["target_mass_kg"]

regime_thresholds = {
    "bmf_low": float(analysis_df["pred_bound_mass_fraction"].quantile(1 / 3)),
    "bmf_high": float(analysis_df["pred_bound_mass_fraction"].quantile(2 / 3)),
    "n_frag_low": float(analysis_df["pred_n_fragments"].quantile(1 / 3)),
    "n_frag_high": float(analysis_df["pred_n_fragments"].quantile(2 / 3)),
    "largest_frac_low": float(analysis_df["pred_largest_fragment_mass_fraction"].quantile(1 / 3)),
    "largest_frac_high": float(analysis_df["pred_largest_fragment_mass_fraction"].quantile(2 / 3)),
}


def classify_regime(bmf, n_frag, largest_frac, thresholds):
    intact_votes = sum([
        bmf >= thresholds["bmf_high"],
        n_frag <= thresholds["n_frag_low"],
        largest_frac >= thresholds["largest_frac_high"],
    ])
    strong_votes = sum([
        bmf <= thresholds["bmf_low"],
        n_frag >= thresholds["n_frag_high"],
        largest_frac <= thresholds["largest_frac_low"],
    ])
    if intact_votes >= 2:
        return "Mostly intact"
    if strong_votes >= 2:
        return "Strong fragmentation"
    return "Moderate fragmentation"


analysis_df["pred_fragmentation_regime"] = analysis_df.apply(
    lambda row: classify_regime(
        row["pred_bound_mass_fraction"],
        row["pred_n_fragments"],
        row["pred_largest_fragment_mass_fraction"],
        regime_thresholds,
    ),
    axis=1,
)
analysis_df["actual_fragmentation_regime"] = analysis_df.apply(
    lambda row: classify_regime(
        row["bound_mass_fraction"],
        row["n_fragments"],
        row["actual_largest_fragment_mass_fraction"],
        regime_thresholds,
    ),
    axis=1,
)

regime_threshold_df = pd.DataFrame([
    {"metric": "predicted bound_mass_fraction", "lower_boundary": regime_thresholds["bmf_low"], "upper_boundary": regime_thresholds["bmf_high"], "basis": "prediction tertiles"},
    {"metric": "predicted n_fragments", "lower_boundary": regime_thresholds["n_frag_low"], "upper_boundary": regime_thresholds["n_frag_high"], "basis": "prediction tertiles"},
    {"metric": "predicted largest_fragment_mass_fraction", "lower_boundary": regime_thresholds["largest_frac_low"], "upper_boundary": regime_thresholds["largest_frac_high"], "basis": "prediction tertiles"},
])

print("Recommended regime boundaries derived from the prediction distributions:")
display(regime_threshold_df.round(6))

print("Predicted regime frequencies:")
display(analysis_df["pred_fragmentation_regime"].value_counts().rename_axis("regime").to_frame("count"))

print("Predicted vs. actual regime labels using the same data-derived thresholds:")
display(pd.crosstab(analysis_df["actual_fragmentation_regime"], analysis_df["pred_fragmentation_regime"], normalize="index").round(3))

fig, ax = plt.subplots(figsize=(8, 6))
regime_colors = {
    "Mostly intact": "seagreen",
    "Moderate fragmentation": "goldenrod",
    "Strong fragmentation": "firebrick",
}
for label, group in analysis_df.groupby("pred_fragmentation_regime"):
    ax.scatter(group["pred_bound_mass_fraction"], group["pred_n_fragments"], s=26, alpha=0.65, color=regime_colors[label], label=label)
ax.axvline(regime_thresholds["bmf_low"], color="black", ls=":", lw=1)
ax.axvline(regime_thresholds["bmf_high"], color="black", ls=":", lw=1)
ax.axhline(regime_thresholds["n_frag_low"], color="black", ls="--", lw=1)
ax.axhline(regime_thresholds["n_frag_high"], color="black", ls="--", lw=1)
ax.set_xlabel("Predicted bound_mass_fraction")
ax.set_ylabel("Predicted n_fragments")
ax.set_title("Fragmentation regimes from regression outputs")
ax.legend()
plt.show()


---
## 9 · SPH Decision Support

This section replaces the previous binary classification gate. The objective is to decide when the ML predictions are sufficient for screening and when a new scenario should still be sent to SPH.

The recommendation uses:

- predicted `bound_mass_fraction`
- predicted fragmentation regime
- distance from the `BMF = 10%` decision boundary
- model spread across the three primary regressors as a simple uncertainty proxy
- the observed OOF error distribution of the default `bound_mass_fraction` model


In [ ]:
primary_model_spread = pd.concat(
    [preds["predicted"].rename(model_name) for model_name, preds in primary_preds.items()],
    axis=1,
).std(axis=1)
analysis_df["pred_bound_mass_fraction_model_spread"] = primary_model_spread
analysis_df["decision_distance"] = analysis_df["pred_bound_mass_fraction"] - BMF_DECISION_THRESHOLD
analysis_df["threshold_region"] = np.select(
    [
        analysis_df["pred_bound_mass_fraction"] < BMF_DECISION_THRESHOLD - bmf_error_margin,
        analysis_df["pred_bound_mass_fraction"] > BMF_DECISION_THRESHOLD + bmf_error_margin,
    ],
    ["Predicted BMF < 10%", "Predicted BMF ≥ 10%"],
    default="Borderline around 10%",
)

spread_threshold = float(analysis_df["pred_bound_mass_fraction_model_spread"].quantile(0.75))
regime_error_summary = (
    analysis_df.groupby("pred_fragmentation_regime")["bmf_abs_error"]
    .agg(["count", "mean", "median", lambda s: s.quantile(0.75)])
    .rename(columns={"<lambda_0>": "q75_abs_error"})
)


def recommend_sph(row):
    reasons = []
    near_boundary = abs(row["decision_distance"]) <= bmf_error_margin
    high_spread = row["pred_bound_mass_fraction_model_spread"] >= spread_threshold
    regime = row["pred_fragmentation_regime"]

    if regime == "Strong fragmentation":
        reasons.append("predicted regime is strongly fragmented, so fine-scale physical outcomes are likely to matter")
    if near_boundary:
        reasons.append(
            f"predicted bound_mass_fraction lies within the data-derived borderline band around {BMF_DECISION_THRESHOLD:.2f}"
        )
    if high_spread:
        reasons.append("the three regression models disagree more than usual on bound mass fraction")

    if regime == "Strong fragmentation":
        recommendation = "Full SPH required for detailed physical outcomes"
    elif near_boundary or high_spread or regime == "Moderate fragmentation":
        recommendation = "Borderline case, SPH recommended"
    else:
        recommendation = "ML prediction sufficient (high confidence)"

    if not reasons:
        reasons.append("prediction is far from the main decision boundary and fragmentation indicators are internally consistent")

    return recommendation, "; ".join(reasons)


analysis_df[["sph_recommendation", "recommendation_reason"]] = analysis_df.apply(
    lambda row: pd.Series(recommend_sph(row)),
    axis=1,
)

decision_thresholds_df = pd.DataFrame([
    {
        "quantity": "BMF screening boundary",
        "value": BMF_DECISION_THRESHOLD,
        "derivation": "research question boundary used for post-regression threshold analysis",
    },
    {
        "quantity": "borderline BMF margin",
        "value": bmf_error_margin,
        "derivation": "75th percentile of OOF absolute error for the default BMF regressor",
    },
    {
        "quantity": "high-uncertainty spread threshold",
        "value": spread_threshold,
        "derivation": "75th percentile of cross-model prediction spread for BMF",
    },
])

print("Decision thresholds derived from the regression results:")
display(decision_thresholds_df.round(6))

print("OOF error summary by predicted fragmentation regime:")
display(regime_error_summary.round(6))

print("Recommendation counts:")
display(analysis_df["sph_recommendation"].value_counts().rename_axis("recommendation").to_frame("count"))


In [ ]:
example_frames = []

ml_examples = (
    analysis_df[analysis_df["sph_recommendation"] == "ML prediction sufficient (high confidence)"]
    .sort_values(["pred_bound_mass_fraction_model_spread", "bmf_abs_error"], ascending=[True, True])
    .head(2)
)
example_frames.append(ml_examples)

borderline_examples = (
    analysis_df[analysis_df["sph_recommendation"] == "Borderline case, SPH recommended"]
    .assign(distance_abs=lambda d: d["decision_distance"].abs())
    .sort_values(["distance_abs", "pred_bound_mass_fraction_model_spread"], ascending=[True, True])
    .head(2)
)
example_frames.append(borderline_examples)

full_examples = (
    analysis_df[analysis_df["sph_recommendation"] == "Full SPH required for detailed physical outcomes"]
    .sort_values(["pred_n_fragments", "pred_bound_mass_fraction"], ascending=[False, True])
    .head(2)
)
example_frames.append(full_examples)

example_df = pd.concat(example_frames).drop_duplicates().reset_index(drop=True)
example_df.insert(0, "example_id", [f"case_{i+1}" for i in range(len(example_df))])

example_display = example_df[[
    "example_id",
    "physical_file",
    "mass_log10_kg",
    "particle_log10",
    "periapsis_Rm",
    "v_inf_kms",
    "spin_period_hr",
    "spin_axis",
    "fof_linking_length",
    "pred_bound_mass_fraction",
    "pred_fragmentation_regime",
    "pred_n_fragments",
    "pred_largest_fragment_mass_kg",
    "pred_bound_fragment_count",
    "pred_largest_bound_fragment_mass_kg",
    "sph_recommendation",
    "recommendation_reason",
]].copy()
example_display = example_display.rename(columns={
    "physical_file": "simulation_group",
    "mass_log10_kg": "input_mass_log10_kg",
    "particle_log10": "input_particle_log10",
    "periapsis_Rm": "input_periapsis_Rm",
    "v_inf_kms": "input_v_inf_kms",
    "spin_period_hr": "input_spin_period_hr",
    "spin_axis": "input_spin_axis",
    "fof_linking_length": "input_fof_linking_length",
    "pred_bound_mass_fraction": "predicted_bound_mass_fraction",
    "pred_fragmentation_regime": "predicted_fragmentation_regime",
    "pred_n_fragments": "predicted_n_fragments",
    "pred_largest_fragment_mass_kg": "predicted_largest_fragment_mass_kg",
    "pred_bound_fragment_count": "predicted_bound_fragment_count",
    "pred_largest_bound_fragment_mass_kg": "predicted_largest_bound_fragment_mass_kg",
})

print("Representative decision-support examples based on OOF-style predictions:")
display(example_display.round(6))


---
## 10 · Parameter-Space Condition Analysis

This section shifts the interpretation from outcome thresholds alone to the **input-parameter conditions** associated with those outcomes. It summarises the simulation setups linked to:

- `predicted_fragmentation_regime`
- `threshold_region`
- `sph_recommendation`

The goal is to identify which combinations of mass, periapsis, velocity, spin, resolution, and FoF settings are most associated with fragmentation, bound retention above 10%, and the need for SPH.


In [ ]:
condition_groups = [
    "pred_fragmentation_regime",
    "threshold_region",
    "sph_recommendation",
]

parameter_space_summaries = {}
for group_col in condition_groups:
    numeric_summary, categorical_summary = summarise_parameter_space(analysis_df, group_col)
    parameter_space_summaries[group_col] = {
        "numeric": numeric_summary,
        "categorical": categorical_summary,
    }

    print("=" * 72)
    print(f"PARAMETER RANGES BY {group_col.upper()}")
    print("=" * 72)
    display(numeric_summary.round(4))

    print()
    print(f"Categorical summaries for {group_col}:")
    display(categorical_summary.round(4))
    print()


In [ ]:
plot_parameter_pairs(analysis_df, "pred_fragmentation_regime", "Fragmentation regime")
plot_parameter_pairs(analysis_df, "sph_recommendation", "SPH recommendation")
plot_parameter_pairs(analysis_df, "threshold_region", "BMF threshold region")


In [ ]:
importance_rows = []
importance_tables = {}

importance_targets = [PRIMARY_TARGET] + FRAGMENTATION_TARGETS
for target in importance_targets:
    model_name = default_models[target]
    if target == PRIMARY_TARGET:
        fitted_model = primary_trained[model_name]
    else:
        fitted_model = frag_trained[target][model_name]

    y_eval = pd.to_numeric(df[target], errors="coerce")
    valid_mask = y_eval.notna()
    X_eval = X.loc[valid_mask]
    y_eval = y_eval.loc[valid_mask]

    importance_df = compute_permutation_importance_table(fitted_model, X_eval, y_eval)
    importance_df.insert(0, "target", target)
    importance_df.insert(1, "model", model_name)
    importance_tables[target] = importance_df
    importance_rows.append(importance_df)

    print(f"Top permutation importance features for {target} ({model_name}):")
    display(importance_df.head(10).round(6))
    print()

combined_importance_df = pd.concat(importance_rows, ignore_index=True)

fig, axes = plt.subplots(len(importance_targets), 1, figsize=(10, 4 * len(importance_targets)))
if len(importance_targets) == 1:
    axes = [axes]
for ax, target in zip(axes, importance_targets):
    top_df = importance_tables[target].head(8).iloc[::-1]
    ax.barh(top_df["feature"], top_df["importance_mean"], xerr=top_df["importance_std"], color="slateblue", alpha=0.85)
    ax.set_title(f"Permutation importance — {target} ({default_models[target]})")
    ax.set_xlabel("Mean decrease in R² after permutation")
plt.tight_layout()
plt.show()


In [ ]:
rule_summary_tables = {
    "pred_fragmentation_regime": make_rule_summary(analysis_df, "pred_fragmentation_regime", "pred_fragmentation_regime"),
    "threshold_region": make_rule_summary(analysis_df, "threshold_region", "threshold_region"),
    "sph_recommendation": make_rule_summary(analysis_df, "sph_recommendation", "sph_recommendation"),
}

print("Automatically generated parameter-space rule summaries:")
display(rule_summary_tables["pred_fragmentation_regime"])
display(rule_summary_tables["threshold_region"])
display(rule_summary_tables["sph_recommendation"])

final_rule_summary = pd.DataFrame([
    {
        "interpretation_target": "mostly intact cases",
        "condition_group": "pred_fragmentation_regime",
        "label": "Mostly intact",
        "parameter_summary": rule_summary_tables["pred_fragmentation_regime"].set_index("pred_fragmentation_regime").loc["Mostly intact", "rule_summary"],
    },
    {
        "interpretation_target": "strong fragmentation",
        "condition_group": "pred_fragmentation_regime",
        "label": "Strong fragmentation",
        "parameter_summary": rule_summary_tables["pred_fragmentation_regime"].set_index("pred_fragmentation_regime").loc["Strong fragmentation", "rule_summary"],
    },
    {
        "interpretation_target": "bound retention above 10%",
        "condition_group": "threshold_region",
        "label": "Predicted BMF ≥ 10%",
        "parameter_summary": rule_summary_tables["threshold_region"].set_index("threshold_region").loc["Predicted BMF ≥ 10%", "rule_summary"],
    },
    {
        "interpretation_target": "SPH recommended",
        "condition_group": "sph_recommendation",
        "label": "Borderline case, SPH recommended",
        "parameter_summary": rule_summary_tables["sph_recommendation"].set_index("sph_recommendation").loc["Borderline case, SPH recommended", "rule_summary"],
    },
    {
        "interpretation_target": "full SPH required",
        "condition_group": "sph_recommendation",
        "label": "Full SPH required for detailed physical outcomes",
        "parameter_summary": rule_summary_tables["sph_recommendation"].set_index("sph_recommendation").loc["Full SPH required for detailed physical outcomes", "rule_summary"],
    },
])

display(final_rule_summary)


---
## 11 · Future Work: Interactive Decision Dashboard

A useful next step is an interactive dashboard that accepts new simulation parameters such as:

- target mass
- periapsis
- encounter velocity
- spin period and spin axis
- numerical resolution
- FoF linking length
- timestep or other setup metadata

The dashboard would then:

1. run the trained regression models
2. estimate `bound_mass_fraction` and the supporting bound metrics
3. predict fragmentation descriptors and assign a fragmentation regime
4. evaluate proximity to the main screening boundary and the model uncertainty proxies
5. recommend whether the case is suitable for **ML-only screening**, should be treated as **borderline**, or should proceed to **full SPH**

The intended purpose is a **fast screening tool before expensive SPH simulations**, not a replacement for SPH when detailed physical outcomes are needed.


---
## 12 · Summary dashboard

This final dashboard consolidates the main regression results and the decision-support framing.


In [ ]:
primary_summary = primary_metrics_df[["model", "r2", "mae", "rmse"]].copy()
primary_summary.insert(0, "target", PRIMARY_TARGET)

summary_metrics_df = pd.concat([
    primary_summary,
    bound_support_metrics_df[["target", "model", "r2", "mae", "rmse"]],
    frag_metrics_df[["target", "model", "r2", "mae", "rmse"]],
], ignore_index=True)

print("=" * 72)
print("PRIMARY REGRESSION")
print("=" * 72)
display(primary_metrics_df[["model", "r2", "mae", "rmse", "train_r2", "train_mae", "train_rmse"]].round(4))

print()
print("=" * 72)
print("SUPPORTING BOUND REGRESSIONS (OOF R²)")
print("=" * 72)
display(bound_support_metrics_df.pivot(index="model", columns="target", values="r2").round(3))

print()
print("=" * 72)
print("FRAGMENTATION REGRESSIONS (OOF R²)")
print("=" * 72)
display(frag_metrics_df.pivot(index="model", columns="target", values="r2").round(3))

print()
print("=" * 72)
print("DEFAULT MODELS USED FOR INTERPRETATION")
print("=" * 72)
display(pd.DataFrame(sorted(default_models.items()), columns=["target", "default_model"]))

print()
print("=" * 72)
print("SCREENING TAKEAWAYS")
print("=" * 72)
print(f"Default BMF baseline model: {default_primary_model}")
print(f"BMF decision threshold analysed after regression: {BMF_DECISION_THRESHOLD:.2f}")
print(f"Borderline margin derived from OOF BMF error: ±{bmf_error_margin:.4f}")
print("Fragmentation regimes are derived from the predicted BMF, fragment-count, and largest-fragment-mass distributions.")
print("SPH recommendations combine regime type, distance to the BMF threshold, and cross-model spread.")
print("Final interpretation should be read in parameter space: which simulation setups map to intact outcomes, fragmentation, or SPH-required cases.")


---
## Notes

- The notebook now treats `bound_mass_fraction` as the **primary/default ML task**.
- No separate classifiers are trained for `has_any_bound_mass`, `BMF > 10%`, or `BMF ≥ 10%`.
- Threshold analysis is performed **after regression** using the predicted continuous `bound_mass_fraction` values.
- Grouped cross-validation still splits by `physical_file` to reduce leakage across related simulations.
- `largest_fragment_mass_kg` remains engineered as `max(largest_bound_fragment_mass_kg, largest_unbound_fragment_mass_kg)` because the raw dataset does not store a direct overall-largest-fragment field.
- `average_bound_fragment_mass_kg` is `bound_mass_kg / bound_fragment_count`; rows with zero bound fragments are excluded from that target via `NaN`.
- `total_fragment_mass_kg` is intentionally excluded from the main fragmentation modelling section because it is less useful for scientific screening than regime-defining fragmentation descriptors.
